# Queries

[GEO query for Inclusion Body Myositis](https://www.ncbi.nlm.nih.gov/gds?term=%28%22Inclusion%20Body%20Myositis%22%5BAll%20Fields%5D%20OR%20%22IBM%22%5BAll%20Fields%5D%29%20AND%20%22Homo%20sapiens%22%5Bporgn%5D%20AND%20%22gse%22%5BFilter%5D%20AND%20%28%22Expression%20profiling%20by%20high%20throughput%20sequencing%22%5BFilter%5D%20OR%20%22Non-coding%20RNA%20profiling%20by%20high%20throughput%20sequencing%22%5BFilter%5D%20OR%20%22Expression%20profiling%20by%20array%22%5BFilter%5D%20OR%20%22Non-coding%20RNA%20profiling%20by%20array%22%5BFilter%5D%29&cmd=DetailsSearch)

Filters:
`("Inclusion Body Myositis"[All Fields] OR "IBM"[All Fields]) AND "Homo sapiens"[porgn] AND "gse"[Filter] AND ("Expression profiling by high throughput sequencing"[Filter] OR "Non-coding RNA profiling by high throughput sequencing"[Filter] OR "Expression profiling by array"[Filter] OR "Non-coding RNA profiling by array"[Filter])`



# Requirements

In [1]:
# Dependencies
!pip3 install rdflib
import os
import json
import rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 8.9 MB/s eta 0:00:00


In [2]:
# Function to query a FAIR Data Point index

def query_FDP(FDP_URL, prefixes = "", graphPattern = "", ordering = "ASC(?title)"):
  # Format input parameters
  URL = FDP_URL.rstrip("/")
  query = {"prefixes": prefixes,
           "graphPattern": graphPattern,
           "ordering": ordering}
  data = json.dumps(query)

  # Create and send request to FAIR Data Point
  request = f'''curl -X 'POST' '{URL}/search/query' \
                -H 'accept: application/json' \
                -H 'Content-Type: application/json' \
                -d '{data}' '''
  response = json.loads(os.popen(request).read())

  # return response
  return(response)

In [3]:
# Test query
query_FDP("https://index.vp.ejprarediseases.org")

[{'uri': 'http://fairdatapointorphanet.info/biobank/563bb462-3e54-46b6-a7af-800237ad54aa',
  'types': ['http://www.w3.org/ns/dcat#Resource',
   'https://w3id.org/ejp-rd/vocabulary#Biobank'],
  'title': 'A biobank of patients with Primary Immune Deficiencies (PID)',
  'description': 'Biobank of A biobank of patients with Primary Immune Deficiencies (PID)',
  'relations': []},
 {'uri': 'http://fairdatapointorphanet.info/patientregistry/b1b271fd-6a9d-47c0-9cbc-1a498b621953',
  'types': ['http://www.w3.org/ns/dcat#Resource',
   'https://w3id.org/ejp-rd/vocabulary#PatientRegistry'],
  'title': 'ADOReg : Nationwide prospective registry for health services research in dermatologic oncology - sub registry Cutaneous Lymphoma',
  'description': 'PatientRegistry of ADOReg : Nationwide prospective registry for health services research in dermatologic oncology - sub registry Cutaneous Lymphoma',
  'relations': []},
 {'uri': 'http://fairdatapointorphanet.info/patientregistry/cadaa706-c0cc-4278-aaf7-

# Query 1
Find datasets with Inclusion Body Myositis as their theme (http://purl.obolibrary.org/obo/DOID_3429)

In [4]:
prefixes = """
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX ejprd: <https://w3id.org/ejp-rd/vocabulary#>
"""

graphPattern = """
?entity a dcat:Dataset .
?entity dcat:theme <http://purl.obolibrary.org/obo/DOID_3429> .
?entity ejprd:vpConnection ejprd:VPDiscoverable .
"""

results = query_FDP("https://index.vp.ejprarediseases.org",
                    prefixes = prefixes,
                    graphPattern = graphPattern)

for result in results:
  print(result["title"] + ": " + result["uri"])

IBM Gene Expression Normalized: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/25e903e5-a771-4aaf-a341-7727e4eb7bdd
IBM Gene Expression Raw: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/4a0cd0b2-baf5-4831-917f-6a68981c709a
IBM MicroRNA Expression Normalized: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/aaed392c-2cbe-42ba-8ada-29e8763e0789
IBM MicroRNA Expression Raw: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/1ab241f1-be2b-4690-b746-888aba0dd3d3
IBM Sample Information: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/956e29d2-5714-482c-8fd6-a50e3fdedb93
IBM Whole Exome Sequencing: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/dfd67457-27cb-48db-a30d-80f0da56f5a4


# Query 2
Find tools that have an input type that matches a theme for the "IBM Gene Expression Raw" dataset (https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/4a0cd0b2-baf5-4831-917f-6a68981c709a) that was found in query 1.

In [5]:
prefixes = """
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX ejprd: <https://w3id.org/ejp-rd/vocabulary#>
PREFIX edam: <http://edamontology.org/>
"""

graphPattern = """
<https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/4a0cd0b2-baf5-4831-917f-6a68981c709a> dcat:theme ?dataset_theme .

?entity a edam:operation_0004 .
?entity edam:has_input ?dataset_theme .
"""

results = query_FDP("https://index.vp.ejprarediseases.org",
                    prefixes = prefixes,
                    graphPattern = graphPattern)

for result in results:
  print(result["title"] + ": " + result["uri"])

DESeq2: https://patient-registries.fdps.ejprd.semlab-leiden.nl/tool/16159983-3703-4255-9065-a57226fe80e7


# Query 3
Find datasets for diseases that are a subclass of Myositis

In [6]:
# Load disease ontology
g = rdflib.Graph()
g.parse("https://raw.githubusercontent.com/DiseaseOntology/HumanDiseaseOntology/main/src/ontology/doid.owl")

<Graph identifier=N56818cdd999249189ca48ecbc1bb168a (<class 'rdflib.graph.Graph'>)>

Find subclasses of myositis in the disease ontology

In [7]:
query = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?subclass  ?label WHERE {
  ?subclass rdfs:subClassOf+ <http://purl.obolibrary.org/obo/DOID_633> .
  ?subclass rdfs:label ?label .
}
"""

qres = g.query(query)
subclasses = " ".join([f"<{row.subclass}>" for row in qres])
labels = "\n".join([f"{row.label}" for row in qres])

print(subclasses)
print(labels)


<http://purl.obolibrary.org/obo/DOID_0080745> <http://purl.obolibrary.org/obo/DOID_10223> <http://purl.obolibrary.org/obo/DOID_14202> <http://purl.obolibrary.org/obo/DOID_14203> <http://purl.obolibrary.org/obo/DOID_3428> <http://purl.obolibrary.org/obo/DOID_3429> <http://purl.obolibrary.org/obo/DOID_668> <http://purl.obolibrary.org/obo/DOID_876> <http://purl.obolibrary.org/obo/DOID_971> <http://purl.obolibrary.org/obo/DOID_10471> <http://purl.obolibrary.org/obo/DOID_10810> <http://purl.obolibrary.org/obo/DOID_14181> <http://purl.obolibrary.org/obo/DOID_970> <http://purl.obolibrary.org/obo/DOID_14192> <http://purl.obolibrary.org/obo/DOID_312> <http://purl.obolibrary.org/obo/DOID_9788>
polymyositis
dermatomyositis
adult dermatomyositis
childhood type dermatomyositis
granulomatous myositis
inclusion body myositis
myositis ossificans
pyomyositis
tendinitis
patellar tendinitis
tibialis tendinitis
calcific tendinitis
tenosynovitis
bicipital tenosynovitis
tenosynovitis of foot and ankle
myosi

Find datasets for any disease that is part of the myositis class

In [9]:
prefixes = """
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX ejprd: <https://w3id.org/ejp-rd/vocabulary#>
"""

graphPattern = f"""
?entity a dcat:Dataset .
?entity dcat:theme ?theme .
VALUES ?theme {{{subclasses}}}
?entity ejprd:vpConnection ejprd:VPDiscoverable .
"""

results = query_FDP("https://index.vp.ejprarediseases.org",
                    prefixes = prefixes,
                    graphPattern = graphPattern)

for result in results:
  print(result["title"] + ": " + result["uri"])

IBM Gene Expression Normalized: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/25e903e5-a771-4aaf-a341-7727e4eb7bdd
IBM Gene Expression Raw: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/4a0cd0b2-baf5-4831-917f-6a68981c709a
IBM MicroRNA Expression Normalized: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/aaed392c-2cbe-42ba-8ada-29e8763e0789
IBM MicroRNA Expression Raw: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/1ab241f1-be2b-4690-b746-888aba0dd3d3
IBM Sample Information: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/956e29d2-5714-482c-8fd6-a50e3fdedb93
IBM Whole Exome Sequencing: https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/dfd67457-27cb-48db-a30d-80f0da56f5a4
